# EHM Fleet Analytics — Phase 4: Remaining Useful Life (RUL) Prediction
### Beginner-friendly version with full explanations
**Author:** Shami | **Platform:** Google Colab | **Version:** v2 (all audit fixes applied)

---

## What this notebook does

Phases 2 and 3 told us **what** the fleet looks like and **when** anomalies occur.
Phase 4 asks: **how many cycles does each engine have left?**

This is called **Remaining Useful Life (RUL) prediction** — the core capability of a modern Prognostics and Health Management (PHM) system.

We build two models and compare them across five analytical questions:

| Model | Type | Why include it |
|---|---|---|
| **XGBoost** | Gradient-boosted trees | Strong on tabular data, fast to train, interpretable via SHAP |
| **LSTM** | Recurrent neural network | Designed for time-series, can remember patterns across many cycles |

---

## Five analytical questions

| # | Question | Why it matters |
|---|---|---|
| **Q1** | How accurately do XGBoost and LSTM predict RUL — and where does accuracy break down? | Accuracy is the foundation. Where it breaks down reveals more than the headline score. |
| **Q2** | In the 200 cycles after degradation starts, does LSTM track RUL more accurately than XGBoost? | This is where sequential memory should matter most. |
| **Q3** | Across the fleet, which features drive RUL predictions? Do they match the physics? | SHAP fleet-level — connects model behaviour to domain knowledge. |
| **Q4** | For each engine, which features dominate? Can we rank engines by fault signature? | SHAP per-engine — a diagnostic triage tool for maintenance controllers. |
| **Q5** | Do engines that start with lower EGT margin degrade faster? | Does starting condition predict degradation trajectory? |

---

## Three design decisions applied in this notebook

Confirmed before Phase 4 began (Handover v2.0, Section 7):

**Decision 1 — Piecewise RUL cap with engine-specific IDP**
Before an engine's CUSUM alarm fires, RUL is capped at the value at that alarm cycle. After the alarm, RUL counts down normally. Improves on Saxena (2008): cap is physically motivated and engine-specific.

**Decision 2 — post_IDP binary feature**
A column set to 0 before the CUSUM alarm and 1 after. Tells the model explicitly that degradation has started.

**Decision 3 — Baseline normalisation per engine**
Raw sensor features normalised against each engine's own first-200-cycle baseline.
Formula: `normalised = (value - baseline_mean) / baseline_std`

---

## How to run
Run cells **top to bottom only**. Do not skip cells. Use **Shift + Enter** to run each cell.

---

## Section 0 — Install and load libraries

In [ ]:
# Install libraries not pre-installed in Colab
!pip install xgboost shap --quiet
print('Installation complete.')

In [ ]:
# --- Load all libraries ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
import shap

from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Colour scheme — consistent with Phases 2 and 3
RED    = '#ef5350'
AMBER  = '#ffa726'
GREEN  = '#66bb6a'
BLUE   = '#42a5f5'
PURPLE = '#ab47bc'
TEAL   = '#26c6da'

plt.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor']   = '#1a1d2e'
plt.rcParams['grid.alpha']       = 0.3

# Fix random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print('All libraries loaded. Ready to begin Phase 4.')

---
## Section 1 — Upload and prepare the data

In [ ]:
# Upload the CSV file
from google.colab import files
uploaded = files.upload()

In [ ]:
# Load and sort the dataset
df = pd.read_csv('ehm_synthetic_fleet_v3.csv')
df = df.sort_values(['engine_id', 'cycle']).reset_index(drop=True)

# --- Audit 2 Finding A2-F1: EGT dropout flag ---
# 100 rows have a missing raw EGT reading (NaN) — sensor dropout events, not a bug
# We flag these as a binary column so the model can use absence as a signal
df['EGT_dropout'] = df['EGT'].isna().astype(int)

print(f'Rows loaded:             {len(df):,}')
print(f'Engines:                 {df["engine_id"].nunique()}')
print(f'EGT dropout rows:        {df["EGT_dropout"].sum()}')
print(f'RUL range:               {df["RUL"].min():,} to {df["RUL"].max():,} cycles')
print(f'Known anomaly rows:      {df["is_anomaly"].sum()}')

all_engines = sorted(df['engine_id'].unique())

---
## Section 2 — Feature Engineering

Five steps in order:
1. Regenerate CUSUM alarm cycle per engine
2. Apply Decision 2 — add `post_IDP` binary column
3. Apply Decision 1 — compute piecewise RUL target
4. Apply Decision 3 — baseline normalise Group 1 features per engine
5. Encode categorical columns and assemble the final feature list

In [ ]:
# ============================================================
# STEP 1: Regenerate CUSUM alarm cycle per engine
# Identical parameters to Phase 3 — do not change these
# ============================================================

K_SLACK       = 0.5    # slack parameter — ignore deviations smaller than half a sigma
H_THRESHOLD   = 5.0    # alarm fires when CUSUM score exceeds this value
BASELINE_CYCLES = 200  # number of healthy cycles used to define the normal baseline

# --------------------------------------------------------
# EQUATION REF — CUSUM update rule
# Source: Page, E.S. (1954). Continuous Inspection Schemes.
#         Biometrika, 41(1/2), pp.100-115.
#
# At each cycle i:
#   standardised_i = (baseline_mean - observed_EGT_margin_i) / baseline_std
#   S_i = max(0, S_{i-1} + standardised_i - k)
#
# Alarm fires when S_i > h
#
# Audit 3 verified this on ENG-004 at cycle 6251: S = 5.4247 (matches Excel to 4dp)
# --------------------------------------------------------

cusum_alarm_cycles = {}   # key = engine_id, value = cycle number when alarm first fired
cusum_score_dict   = {}   # full CUSUM score series per engine (used in Q2)

for engine_id in all_engines:

    eng     = df[df['engine_id'] == engine_id].copy()
    margins = eng['EGT_margin'].values

    baseline_mean = margins[:BASELINE_CYCLES].mean()
    baseline_std  = margins[:BASELINE_CYCLES].std()
    if baseline_std == 0:
        baseline_std = 0.001

    cusum_scores = []
    s = 0

    for margin_value in margins:
        standardised = (baseline_mean - margin_value) / baseline_std
        s = max(0, s + standardised - K_SLACK)
        cusum_scores.append(s)

    cusum_score_dict[engine_id] = pd.Series(cusum_scores, index=eng['cycle'].values)

    alarm_indices = [i for i, sc in enumerate(cusum_scores) if sc > H_THRESHOLD]
    if alarm_indices:
        alarm_cycle = eng['cycle'].values[alarm_indices[0]]
    else:
        alarm_cycle = eng['cycle'].median()

    cusum_alarm_cycles[engine_id] = alarm_cycle

print('CUSUM alarm cycles regenerated for all engines.')
print()
print(f'  {"Engine":<12} {"CUSUM alarm cycle":>18}')
print('  ' + '-' * 32)
for eid, acycle in sorted(cusum_alarm_cycles.items()):
    print(f'  {eid:<12} {int(acycle):>18,}')

In [ ]:
# ============================================================
# STEP 2: Apply Decision 2 — add post_IDP binary column
# Design Decision 2, Handover v2.0 Section 7
# ============================================================

# post_IDP = 0 before the CUSUM alarm fires for this engine
# post_IDP = 1 from the CUSUM alarm cycle onwards
#
# AUDIT 4 CHECK A4-3:
# For any engine, the first row where post_IDP = 1 should equal cusum_alarm_cycles for that engine
# Verification print below confirms this for ENG-004

df['post_IDP'] = 0

for engine_id in all_engines:
    alarm_cycle = cusum_alarm_cycles[engine_id]
    mask = (df['engine_id'] == engine_id) & (df['cycle'] >= alarm_cycle)
    df.loc[mask, 'post_IDP'] = 1

# Verification
eng4_first_post = df[(df['engine_id'] == 'ENG-004') & (df['post_IDP'] == 1)]['cycle'].min()
eng4_alarm      = cusum_alarm_cycles['ENG-004']
match_str = 'MATCH' if eng4_first_post == eng4_alarm else 'MISMATCH — CHECK CODE'

print(f'post_IDP column created.')
print(f'  Rows before IDP (post_IDP=0): {(df["post_IDP"]==0).sum():,}')
print(f'  Rows after IDP  (post_IDP=1): {(df["post_IDP"]==1).sum():,}')
print()
print(f'AUDIT 4 — A4-3: post_IDP flag alignment check (ENG-004)')
print(f'  post_IDP switches at cycle:   {int(eng4_first_post):,}')
print(f'  CUSUM alarm stored as:        {int(eng4_alarm):,}')
print(f'  Result: {match_str}')

In [ ]:
# ============================================================
# STEP 3: Apply Decision 1 — Piecewise RUL target
# Design Decision 1, Handover v2.0 Section 7
# Source: Saxena et al. (2008), Section 4
# ============================================================

# --------------------------------------------------------
# EQUATION REF — Piecewise RUL
# Source: Saxena, A. et al. (2008). Damage Propagation Modeling for
#         Aircraft Engine Run-to-Failure Simulation. PHM Conference.
#         Available at ntrs.nasa.gov
#
#   RUL_target_i =
#       RUL_cap   if cycle_i < CUSUM_alarm_cycle  (pre-IDP: cap at alarm-cycle RUL)
#       RUL_i     if cycle_i >= CUSUM_alarm_cycle (post-IDP: actual countdown)
#
# AUDIT 4 CHECK A4-2:
# All pre-IDP rows for an engine should have the same RUL_target value (the cap)
# Post-IDP rows should have RUL_target = RUL column
# Verification print below confirms this for ENG-004
# --------------------------------------------------------

df['RUL_target'] = df['RUL'].copy()

for engine_id in all_engines:

    alarm_cycle = cusum_alarm_cycles[engine_id]

    alarm_row = df[(df['engine_id'] == engine_id) & (df['cycle'] == alarm_cycle)]
    if len(alarm_row) == 0:
        alarm_row = df[df['engine_id'] == engine_id].iloc[
            (df[df['engine_id'] == engine_id]['cycle'] - alarm_cycle).abs().argsort()[:1]
        ]

    rul_cap = alarm_row['RUL'].values[0]

    pre_idp_mask = (df['engine_id'] == engine_id) & (df['cycle'] < alarm_cycle)
    df.loc[pre_idp_mask, 'RUL_target'] = rul_cap

# Verification on ENG-004
eng4       = df[df['engine_id'] == 'ENG-004'].copy()
eng4_alarm = cusum_alarm_cycles['ENG-004']
eng4_cap   = eng4[eng4['cycle'] == eng4_alarm]['RUL'].values[0]
eng4_pre_unique = eng4[eng4['cycle'] < eng4_alarm]['RUL_target'].nunique()

print('RUL_target (piecewise) created.')
print(f'  Original RUL range:    {df["RUL"].min():,} to {df["RUL"].max():,}')
print(f'  RUL_target range:      {df["RUL_target"].min():,} to {df["RUL_target"].max():,}')
print()
print('AUDIT 4 — A4-2: Piecewise RUL target check (ENG-004)')
print(f'  CUSUM alarm cycle:              {int(eng4_alarm):,}')
print(f'  RUL cap at alarm:               {int(eng4_cap):,}')
print(f'  Unique RUL_target values pre-IDP: {eng4_pre_unique}  (should be 1)')
print(f'  Pre-IDP sample values:  {eng4[eng4["cycle"] < eng4_alarm]["RUL_target"].values[:3]}')
print(f'  Post-IDP sample values: {eng4[eng4["cycle"] >= eng4_alarm]["RUL_target"].values[-3:]}')

In [ ]:
# ============================================================
# STEP 4: Apply Decision 3 — Baseline normalisation per engine
# Design Decision 3, Handover v2.0 Section 7
# ============================================================

# --------------------------------------------------------
# EQUATION REF — Baseline Z-score normalisation
# Source: Heimes, F.O. (2008). Recurrent Neural Networks for
#         Remaining Useful Life Estimation. PHM Conference.
#
#   normalised_value = (observed_value - baseline_mean) / baseline_std
#
#   baseline_mean and baseline_std computed from first BASELINE_CYCLES
#   rows of THIS engine only.
#
# AUDIT 4 CHECK A4-1:
# Pick ENG-001, EGT feature.
# In Excel: AVERAGE(first 200 EGT values), STDEV(first 200 EGT values)
# Then for row 0: (EGT - mean) / std — should match EGT_norm to 4 decimal places
# Verification numbers printed below
# --------------------------------------------------------

GROUP1_FEATURES = [
    'EGT', 'N1', 'N2', 'fuel_flow', 'EPR',
    'oil_temp', 'oil_pressure', 'oil_consumption',
    'vibration_1', 'vibration_2'
]

baseline_stats = {}

for engine_id in all_engines:

    eng_mask = (df['engine_id'] == engine_id)
    eng_data = df[eng_mask].copy()
    baseline_stats[engine_id] = {}

    for feature in GROUP1_FEATURES:

        baseline_values = eng_data[feature].values[:BASELINE_CYCLES]
        b_mean = np.nanmean(baseline_values)
        b_std  = np.nanstd(baseline_values)

        if b_std == 0 or np.isnan(b_std):
            b_std = 1.0

        baseline_stats[engine_id][feature] = (b_mean, b_std)

        # Apply normalisation — NaN raw values stay NaN at this stage
        # They will be filled with zero before LSTM training (Fix 1 below)
        normalised_values = (eng_data[feature].values - b_mean) / b_std
        df.loc[eng_mask, feature + '_norm'] = normalised_values

# Verification spot check — ENG-001, EGT
e1_mean, e1_std   = baseline_stats['ENG-001']['EGT']
e1_raw            = df[df['engine_id'] == 'ENG-001']['EGT'].values[0]
e1_norm_col       = df[df['engine_id'] == 'ENG-001']['EGT_norm'].values[0]
e1_manual         = (e1_raw - e1_mean) / e1_std

print('Baseline normalisation complete.')
print()
print('AUDIT 4 — A4-1: Baseline normalisation check (ENG-001, EGT)')
print(f'  Baseline mean (first 200 cycles): {e1_mean:.4f}')
print(f'  Baseline std  (first 200 cycles): {e1_std:.4f}')
print(f'  Raw EGT at row 0:                 {e1_raw:.4f}')
print(f'  Manual calc (raw - mean) / std:   {e1_manual:.4f}')
print(f'  EGT_norm column at row 0:         {e1_norm_col:.4f}')
print(f'  Match (within 0.0001):            {abs(e1_manual - e1_norm_col) < 0.0001}')
print()
print('To verify in Excel:')
print(f'  Filter ENG-001. Take first 200 EGT values.')
print(f'  =AVERAGE(...) should give {e1_mean:.4f}')
print(f'  =STDEV(...)   should give {e1_std:.4f}')
print(f'  For row 0: =({e1_raw:.4f} - {e1_mean:.4f}) / {e1_std:.4f} = {e1_manual:.4f}')

In [ ]:
# ============================================================
# STEP 5: Encode categorical columns and assemble feature list
# ============================================================

# One-hot encode route_type and age_bucket
# pd.get_dummies creates one numeric 0/1 column per category
route_dummies = pd.get_dummies(df['route_type'], prefix='route')
age_dummies   = pd.get_dummies(df['age_bucket'],  prefix='age')
df = pd.concat([df, route_dummies, age_dummies], axis=1)

# Convert has_faulty_sensor from True/False to 1/0
df['has_faulty_sensor_flag'] = df['has_faulty_sensor'].astype(int)

# Collect the exact dummy column names created by pd.get_dummies
# (collected dynamically so we never hardcode names that might not exist)
route_cols = [c for c in df.columns if c.startswith('route_')]
age_cols   = [c for c in df.columns if c.startswith('age_')]

print('Dummy columns created:')
print(f'  Route: {route_cols}')
print(f'  Age:   {age_cols}')
print()

# --- Define the final feature list for XGBoost ---

# Group 1: baseline-normalised raw sensors
FEATURES_NORM = [f + '_norm' for f in GROUP1_FEATURES]

# Group 2: ISA-corrected features — already normalised by physics
FEATURES_ISA = ['EGT_corrected', 'N1_corrected', 'N2_corrected',
                'fuel_flow_corrected', 'theta', 'delta']

# Group 3: binary and one-hot encoded
# NOTE: route_type and age_bucket original text columns are deliberately
# excluded here — only the numeric dummy columns are included
FEATURES_CAT = (
    ['post_IDP', 'has_faulty_sensor_flag', 'EGT_dropout']
    + route_cols
    + age_cols
)

# Group 4: degradation columns
FEATURES_DEG = ['HPT_degradation', 'HPC_degradation',
                'LPT_degradation', 'fan_degradation', 'SFC_degradation_pct']

# Group 5: contextual features
FEATURES_CTX = ['cruise_alt_ft', 'OAT_C', 'thrust_pct', 'cruise_mach']

ALL_FEATURES = FEATURES_NORM + FEATURES_ISA + FEATURES_CAT + FEATURES_DEG + FEATURES_CTX

TARGET = 'RUL_target'

# --- Safety check: verify no text columns slipped into ALL_FEATURES ---
X_check     = df[ALL_FEATURES]
object_cols = X_check.select_dtypes(include='object').columns.tolist()
if object_cols:
    print(f'WARNING — text columns still present: {object_cols}')
else:
    print('All features are numeric. Safe to train XGBoost.')

print()
print(f'Feature set assembled:')
print(f'  Group 1 (normalised raw sensors):  {len(FEATURES_NORM)}')
print(f'  Group 2 (ISA-corrected):           {len(FEATURES_ISA)}')
print(f'  Group 3 (categorical / binary):    {len(FEATURES_CAT)}')
print(f'  Group 4 (degradation):             {len(FEATURES_DEG)}')
print(f'  Group 5 (contextual):              {len(FEATURES_CTX)}')
print(f'  Total features for XGBoost:        {len(ALL_FEATURES)}')

---
## Section 3 — Train/Test Split

We hold out **entire engines** for testing — not random rows. The model must predict RUL for engines it has never seen. That is the real-world task.

Source for this methodology: Saxena et al. (2008), Section 3.

In [ ]:
# Hold out 4 complete engines: 2 RED, 1 AMBER, 1 GREEN
# The 2 RED engines (ENG-005, ENG-017) are the hardest test —
# the model must generalise to end-of-life without ever seeing a RED engine in training

TEST_ENGINES  = ['ENG-005', 'ENG-017', 'ENG-001', 'ENG-002']
TRAIN_ENGINES = [e for e in all_engines if e not in TEST_ENGINES]

train_df = df[df['engine_id'].isin(TRAIN_ENGINES)].copy()
test_df  = df[df['engine_id'].isin(TEST_ENGINES)].copy()

X_train = train_df[ALL_FEATURES]
y_train = train_df[TARGET]
X_test  = test_df[ALL_FEATURES]
y_test  = test_df[TARGET]

print('Train / test split complete (by engine — NOT by row).')
print(f'  Training engines ({len(TRAIN_ENGINES)}): {TRAIN_ENGINES}')
print(f'  Test engines     ({len(TEST_ENGINES)}):  {TEST_ENGINES}')
print(f'  Training rows:   {len(train_df):,}')
print(f'  Test rows:       {len(test_df):,}')

---
## Section 4 — Evaluation Metrics

Three metrics defined before training begins — not chosen after seeing results.

- **RMSE** — overall accuracy, penalises large errors
- **MAE** — average absolute error in cycles
- **PHM Score** — asymmetric, late predictions penalised ~11x harder than early ones (Saxena 2008)

In [ ]:
# ============================================================
# Define the three evaluation metrics
# ============================================================

def compute_rmse(y_true, y_pred):
    """
    Root Mean Squared Error.
    EQUATION: RMSE = sqrt( mean( (y_true - y_pred)^2 ) )
    Lower is better. Units = cycles.
    """
    return np.sqrt(mean_squared_error(y_true, y_pred))


def compute_mae(y_true, y_pred):
    """
    Mean Absolute Error.
    EQUATION: MAE = mean( |y_true - y_pred| )
    Lower is better. Units = cycles.
    """
    return mean_absolute_error(y_true, y_pred)


def compute_phm_score(y_true, y_pred):
    """
    PHM Scoring Function — asymmetric penalty for late predictions.

    SOURCE: Saxena, A. et al. (2008), Section 5.
    Available at: ntrs.nasa.gov — search 'C-MAPSS turbofan damage propagation'

    EQUATION:
        error_i = y_pred_i - y_true_i

        If error_i < 0 (early — model underestimates remaining life):
            penalty_i = exp(-error_i / 13) - 1

        If error_i >= 0 (late — model overestimates remaining life):
            penalty_i = exp(error_i / 10) - 1

        PHM_score = sum(penalty_i)

    Late errors are ~11x more costly than early errors at equal magnitude.

    NOTE ON CLIPPING:
        exp() overflows to inf for inputs above ~709 in float64.
        We clip the exponent at 500 to prevent numerical crash.
        Any error large enough to trigger clipping is already catastrophically bad.

    AUDIT 4 CHECK A4-4 — verify by hand:
        error = +100 (late):  exp(min(100/10, 500)) - 1 = exp(10)  - 1 = 22,025.47
        error = -100 (early): exp(min(100/13, 500)) - 1 = exp(7.69) - 1 = 1,997.19
        error =    0 (exact): exp(0) - 1 = 0
        Total for these 3 rows = 22,025.47 + 1,997.19 + 0 = 24,022.66
    """
    errors = np.array(y_pred) - np.array(y_true)   # positive = late, negative = early

    penalties = np.where(
        errors < 0,
        np.exp(np.clip(-errors / 13, -500, 500)) - 1,   # early — gentler
        np.exp(np.clip( errors / 10, -500, 500)) - 1    # late  — harsher
    )

    return np.sum(penalties)


# --- AUDIT 4: A4-4 — PHM score formula verification ---
print('AUDIT 4 — A4-4: PHM score formula verification')
print('Three test cases — verify these numbers by hand using the equations above:')
print()
ex_true = [1000, 1000, 1000]
ex_pred = [1100,  900, 1000]   # late, early, exact
running_total = 0
for t, p in zip(ex_true, ex_pred):
    e = p - t
    if e < 0:
        penalty   = np.exp(np.clip(-e / 13, -500, 500)) - 1
        direction = 'early'
    else:
        penalty   = np.exp(np.clip(e / 10, -500, 500)) - 1
        direction = 'late' if e > 0 else 'exact'
    running_total += penalty
    print(f'  y_true={t}, y_pred={p}, error={e:+d} ({direction}) => penalty = {penalty:,.2f}')
print(f'  Total PHM score: {running_total:,.2f}')
print()
print('Expected: late +100 => 22,025.47 | early -100 => 1,997.19 | exact => 0')
print('Metrics defined. Ready to train.')

---
## Section 5 — XGBoost Model

In [ ]:
# ============================================================
# Train the XGBoost model
# ============================================================

xgb_model = xgb.XGBRegressor(
    n_estimators     = 500,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    random_state     = 42,
    verbosity        = 0
)

print('Training XGBoost model...')
print(f'  Training rows: {len(X_train):,}')
print(f'  Features:      {len(ALL_FEATURES)}')

xgb_model.fit(X_train, y_train)

xgb_pred_train = xgb_model.predict(X_train)
xgb_pred_test  = xgb_model.predict(X_test)

xgb_rmse_train = compute_rmse(y_train, xgb_pred_train)
xgb_rmse_test  = compute_rmse(y_test,  xgb_pred_test)
xgb_mae_test   = compute_mae(y_test,   xgb_pred_test)
xgb_phm_test   = compute_phm_score(y_test.values, xgb_pred_test)

print()
print('XGBoost training complete.')
print(f'  RMSE (training set):  {xgb_rmse_train:,.0f} cycles  <- much lower than test = overfit')
print(f'  RMSE (test set):      {xgb_rmse_test:,.0f} cycles')
print(f'  MAE  (test set):      {xgb_mae_test:,.0f} cycles')
print(f'  PHM Score (test set): {xgb_phm_test:,.0f}  (lower = better)')

---
## Section 6 — LSTM Model

LSTM processes sequences of cycles rather than individual rows. Each sample is a window of `WINDOW_SIZE` consecutive cycles. The model learns to predict RUL from that window of history.

Two fixes are applied before training:
- **Fix 1:** NaN values in normalised features are replaced with zero (sensor dropout rows)
- **Fix 2:** RUL target is scaled to [0, 1] range to prevent gradient overflow

In [ ]:
# ============================================================
# Prepare sequence windows for LSTM
# ============================================================

# LSTM uses time-varying features only — excludes one-hot encoded columns
LSTM_FEATURES = FEATURES_NORM + FEATURES_ISA + FEATURES_DEG + FEATURES_CTX + ['post_IDP']

WINDOW_SIZE = 50  # how many consecutive cycles the LSTM looks at per prediction

# --------------------------------------------------------
# EQUATION REF — LSTM sequence window construction
# Source: Heimes, F.O. (2008). Recurrent Neural Networks for
#         Remaining Useful Life Estimation. PHM Conference.
#
# For an engine with N cycles, we create (N - WINDOW_SIZE) samples:
#   X_i = rows [i, i+1, ..., i + WINDOW_SIZE - 1]  shape: (WINDOW_SIZE, n_features)
#   y_i = RUL_target at row [i + WINDOW_SIZE - 1]
# --------------------------------------------------------

def make_windows(engine_df, feature_cols, target_col, window_size):
    """Convert one engine's data into overlapping sequence windows."""
    data   = engine_df[feature_cols].values
    target = engine_df[target_col].values
    X_windows, y_windows = [], []
    for i in range(len(data) - window_size):
        X_windows.append(data[i : i + window_size])
        y_windows.append(target[i + window_size - 1])
    return np.array(X_windows), np.array(y_windows)


# Build training windows
X_lstm_train_list, y_lstm_train_list = [], []
for engine_id in TRAIN_ENGINES:
    eng_data = train_df[train_df['engine_id'] == engine_id]
    X_w, y_w = make_windows(eng_data, LSTM_FEATURES, TARGET, WINDOW_SIZE)
    X_lstm_train_list.append(X_w)
    y_lstm_train_list.append(y_w)

X_lstm_train = np.concatenate(X_lstm_train_list, axis=0)
y_lstm_train = np.concatenate(y_lstm_train_list, axis=0)

# Build test windows — store metadata (engine_id, cycle) for Q2 analysis
X_lstm_test_list, y_lstm_test_list, lstm_test_meta = [], [], []
for engine_id in TEST_ENGINES:
    eng_data = test_df[test_df['engine_id'] == engine_id]
    X_w, y_w = make_windows(eng_data, LSTM_FEATURES, TARGET, WINDOW_SIZE)
    cycles = eng_data['cycle'].values
    for i in range(len(cycles) - WINDOW_SIZE):
        lstm_test_meta.append({'engine_id': engine_id, 'cycle': cycles[i + WINDOW_SIZE - 1]})
    X_lstm_test_list.append(X_w)
    y_lstm_test_list.append(y_w)

X_lstm_test      = np.concatenate(X_lstm_test_list, axis=0)
y_lstm_test      = np.concatenate(y_lstm_test_list, axis=0)
lstm_test_meta_df = pd.DataFrame(lstm_test_meta)

print(f'LSTM sequence windows created.')
print(f'  Window size:            {WINDOW_SIZE} cycles')
print(f'  LSTM features per step: {len(LSTM_FEATURES)}')
print(f'  Training windows:       {X_lstm_train.shape[0]:,}  shape: {X_lstm_train.shape}')
print(f'  Test windows:           {X_lstm_test.shape[0]:,}  shape: {X_lstm_test.shape}')

In [ ]:
# ============================================================
# Diagnostic: check for NaN before applying fixes
# ============================================================

print('NaN diagnostic (before fixes):')
print(f'  NaN in X_lstm_train: {np.isnan(X_lstm_train).sum():,}')
print(f'  NaN in y_lstm_train: {np.isnan(y_lstm_train).sum()}')
print(f'  y_lstm_train range:  {y_lstm_train.min():,.0f} to {y_lstm_train.max():,.0f}')
print()

nan_features = []
for i, feat in enumerate(LSTM_FEATURES):
    n_nan = np.isnan(X_lstm_train[:, :, i]).sum()
    if n_nan > 0:
        nan_features.append((feat, n_nan))
        print(f'  NaN in {feat}: {n_nan:,}')

if not nan_features:
    print('  No NaN found in any feature.')

In [ ]:
# ============================================================
# Fix 1: Replace NaN with zero in LSTM windows
# Fix 2: Scale RUL target to prevent gradient overflow
# ============================================================

# --- FIX 1: NaN -> 0 ---
# NaN values come from sensor dropout rows.
# Zero in a normalised column = same as baseline mean = neutral value.
# This is standard imputation for missing sensor readings in EHM.

X_lstm_train_clean = np.nan_to_num(X_lstm_train, nan=0.0)
X_lstm_test_clean  = np.nan_to_num(X_lstm_test,  nan=0.0)

print(f'Fix 1 applied — NaN after fill: {np.isnan(X_lstm_train_clean).sum()}  (should be 0)')

# --- FIX 2: Scale RUL target ---
# --------------------------------------------------------
# EQUATION REF — Target scaling for neural networks
# Source: Heimes (2008), Section 3 — target normalisation
#
#   y_scaled = y / RUL_MAX_SCALE
#
# We divide by the maximum training RUL so the target is in [0, 1].
# After prediction, we multiply back:
#   y_cycles = y_scaled * RUL_MAX_SCALE
#
# AUDIT 4 — to verify:
#   Pick any y value. Divide by RUL_MAX_SCALE. That is y_scaled.
#   After prediction, multiply by RUL_MAX_SCALE to recover cycles.
# --------------------------------------------------------

RUL_MAX_SCALE = float(y_lstm_train.max())

y_lstm_train_scaled = y_lstm_train / RUL_MAX_SCALE
y_lstm_test_scaled  = y_lstm_test  / RUL_MAX_SCALE

print(f'Fix 2 applied — RUL_MAX_SCALE: {RUL_MAX_SCALE:,.0f} cycles')
print(f'  y_lstm_train_scaled range: {y_lstm_train_scaled.min():.4f} to {y_lstm_train_scaled.max():.4f}')
print(f'  y_lstm_test_scaled range:  {y_lstm_test_scaled.min():.4f} to {y_lstm_test_scaled.max():.4f}')
print()
print('Both fixes applied. Safe to train LSTM.')

In [ ]:
# ============================================================
# Build and train the LSTM model
# ============================================================

# Architecture:
#   LSTM(64) -> Dropout(0.2) -> LSTM(32) -> Dropout(0.2) -> Dense(1)
# Source for two-layer LSTM with dropout:
#   Heimes (2008) used single LSTM; two-layer with dropout is standard
#   in subsequent C-MAPSS literature for improved generalisation.

lstm_model = Sequential([
    LSTM(64, return_sequences=True,
         input_shape=(WINDOW_SIZE, len(LSTM_FEATURES))),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(1)
])

lstm_model.compile(optimizer='adam', loss='mse')
lstm_model.summary()

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

print('Training LSTM model...')

history = lstm_model.fit(
    X_lstm_train_clean, y_lstm_train_scaled,
    epochs           = 100,
    batch_size       = 256,
    validation_split = 0.15,
    callbacks        = [early_stop],
    verbose          = 1
)

In [ ]:
# ============================================================
# Evaluate the LSTM on the test set
# ============================================================

# Check which epoch produced the best validation loss
best_epoch    = np.argmin(history.history['val_loss']) + 1
best_val_loss = min(history.history['val_loss'])
approx_rmse   = np.sqrt(best_val_loss) * RUL_MAX_SCALE

print(f'Best epoch:                    {best_epoch}')
print(f'Best val_loss (scaled):        {best_val_loss:.6f}')
print(f'Approx RMSE in cycles:         {approx_rmse:,.0f}')
print()

# Generate predictions — scale back to cycles before computing metrics
lstm_pred_scaled = lstm_model.predict(X_lstm_test_clean, verbose=0).flatten()
lstm_pred_test   = lstm_pred_scaled * RUL_MAX_SCALE   # convert scaled predictions back to cycles

lstm_rmse_test = compute_rmse(y_lstm_test, lstm_pred_test)
lstm_mae_test  = compute_mae(y_lstm_test,  lstm_pred_test)
lstm_phm_test  = compute_phm_score(y_lstm_test, lstm_pred_test)

print('LSTM evaluation complete.')
print(f'  RMSE (test set):      {lstm_rmse_test:,.0f} cycles')
print(f'  MAE  (test set):      {lstm_mae_test:,.0f} cycles')
print(f'  PHM Score (test set): {lstm_phm_test:,.0f}')

# Training history chart
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history.history['loss'],     color=BLUE,  label='Training loss')
ax.plot(history.history['val_loss'], color=AMBER, label='Validation loss')
ax.set_title('LSTM Training History — Loss per Epoch', fontsize=12)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss (scaled)')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()
print('Both lines should decrease then flatten. If training loss falls but val_loss rises = overfitting.')

---
## Q1 — How accurately do XGBoost and LSTM predict RUL, and where does accuracy break down?

In [ ]:
# Add XGBoost predictions to test_df
test_df = test_df.copy()
test_df['xgb_pred'] = xgb_pred_test

# Add LSTM predictions via the metadata we stored when building windows
lstm_pred_df = lstm_test_meta_df.copy()
lstm_pred_df['lstm_pred'] = lstm_pred_test
test_df = test_df.merge(lstm_pred_df, on=['engine_id', 'cycle'], how='left')

# Label each row with a life stage
def label_life_stage(row):
    if row['post_IDP'] == 0:
        return 'Early (pre-IDP)'
    elif row['RUL_target'] > 2000:
        return 'Mid (post-IDP, RUL>2000)'
    else:
        return 'Late (post-IDP, RUL<=2000)'

test_df['life_stage'] = test_df.apply(label_life_stage, axis=1)

print('=== Q1: RUL Prediction Accuracy by Life Stage ===')
print()
print(f'{"Life Stage":<30} {"N rows":>8} {"XGB MAE":>10} {"LSTM MAE":>10}')
print('-' * 62)

for stage in ['Early (pre-IDP)', 'Mid (post-IDP, RUL>2000)', 'Late (post-IDP, RUL<=2000)']:
    stage_df = test_df[test_df['life_stage'] == stage].dropna(subset=['xgb_pred', 'lstm_pred'])
    if len(stage_df) == 0:
        continue
    xgb_mae  = compute_mae(stage_df['RUL_target'], stage_df['xgb_pred'])
    lstm_mae = compute_mae(stage_df['RUL_target'], stage_df['lstm_pred'])
    print(f'{stage:<30} {len(stage_df):>8,} {xgb_mae:>10,.0f} {lstm_mae:>10,.0f}')

print()
print('(MAE = average absolute error in cycles. Lower = more accurate.)')
print('Late-life accuracy matters most — errors here affect maintenance scheduling directly.')

In [ ]:
# Q1 Chart — Predicted vs Actual RUL for all test engines

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, engine_id in enumerate(TEST_ENGINES):
    ax  = axes[i]
    eng = test_df[test_df['engine_id'] == engine_id].dropna(subset=['xgb_pred'])

    ax.plot(eng['cycle'], eng['RUL_target'],
            color=GREEN, linewidth=2.0, label='True RUL (piecewise)', alpha=0.9)
    ax.plot(eng['cycle'], eng['xgb_pred'],
            color=BLUE, linewidth=1.5, label='XGBoost prediction', alpha=0.7, linestyle='--')

    eng_lstm = eng.dropna(subset=['lstm_pred'])
    ax.plot(eng_lstm['cycle'], eng_lstm['lstm_pred'],
            color=AMBER, linewidth=1.5, label='LSTM prediction', alpha=0.7, linestyle='-.')

    idp_cycle = cusum_alarm_cycles[engine_id]
    ax.axvline(idp_cycle, color=PURPLE, linestyle=':', linewidth=1.5,
               label=f'IDP cycle {int(idp_cycle):,}')

    last = test_df[test_df['engine_id'] == engine_id].iloc[-1]
    rag  = 'RED' if (last['EGT_margin'] < 30 or last['RUL'] < 200) else \
           'AMBER' if (last['EGT_margin'] < 50 or last['RUL'] < 500) else 'GREEN'
    rag_colour = {'RED': RED, 'AMBER': AMBER, 'GREEN': GREEN}[rag]

    ax.set_title(f'Q1: {engine_id} [{rag}]', fontsize=10, color=rag_colour)
    ax.set_xlabel('Flight Cycle')
    ax.set_ylabel('RUL (cycles)')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.2)

plt.suptitle('Q1: XGBoost and LSTM RUL Predictions vs True RUL — Test Engines',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Q1 Full metrics summary
print('=== Q1: Full Metrics Summary ===')
print()
print(f'{"Metric":<25} {"XGBoost":>15} {"LSTM":>15}')
print('-' * 57)
print(f'{"RMSE (cycles)":<25} {xgb_rmse_test:>15,.0f} {lstm_rmse_test:>15,.0f}')
print(f'{"MAE  (cycles)":<25} {xgb_mae_test:>15,.0f} {lstm_mae_test:>15,.0f}')
print(f'{"PHM Score":<25} {xgb_phm_test:>15,.0f} {lstm_phm_test:>15,.0f}')
print()
print('PHM Score: lower = better. Late predictions penalised ~11x harder than early ones.')
print()
print('Honest caveat: synthetic data is cleaner than real EHM data.')
print('Real predictions would show higher errors due to OAT scatter (Assumption A-02).')

---
## Q2 — In the 200 cycles after IDP, does LSTM track RUL more accurately than XGBoost?

In [ ]:
POST_IDP_WINDOW = 200
q2_results = []

for engine_id in TEST_ENGINES:

    alarm_cycle = cusum_alarm_cycles[engine_id]
    eng = test_df[test_df['engine_id'] == engine_id].copy()

    window_mask = (eng['cycle'] >= alarm_cycle) & (eng['cycle'] < alarm_cycle + POST_IDP_WINDOW)
    window_df   = eng[window_mask].dropna(subset=['xgb_pred', 'lstm_pred'])

    if len(window_df) == 0:
        continue

    xgb_mae_w  = compute_mae(window_df['RUL_target'], window_df['xgb_pred'])
    lstm_mae_w = compute_mae(window_df['RUL_target'], window_df['lstm_pred'])

    window_df = window_df.copy()
    window_df['xgb_error']     = window_df['xgb_pred']  - window_df['RUL_target']
    window_df['lstm_error']    = window_df['lstm_pred'] - window_df['RUL_target']
    window_df['cycles_post_idp'] = window_df['cycle'] - alarm_cycle

    q2_results.append(window_df)

    winner = 'LSTM' if lstm_mae_w < xgb_mae_w else 'XGBoost'
    print(f'{engine_id}:  IDP at cycle {int(alarm_cycle):,}  |  '
          f'XGB MAE: {xgb_mae_w:,.0f}  |  LSTM MAE: {lstm_mae_w:,.0f}  |  Winner: {winner}')

q2_df = pd.concat(q2_results, ignore_index=True)

In [ ]:
# Q2 Chart — mean prediction error in the post-IDP window

grouped = q2_df.groupby('cycles_post_idp').agg(
    xgb_mean_error  = ('xgb_error',  'mean'),
    lstm_mean_error = ('lstm_error', 'mean')
).reset_index()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(grouped['cycles_post_idp'], grouped['xgb_mean_error'],
        color=BLUE, linewidth=2.0, label='XGBoost mean error')
ax.plot(grouped['cycles_post_idp'], grouped['lstm_mean_error'],
        color=AMBER, linewidth=2.0, label='LSTM mean error')
ax.axhline(0, color='white', linestyle='--', alpha=0.5, linewidth=1.0, label='Zero error')
ax.fill_between(grouped['cycles_post_idp'], grouped['xgb_mean_error'], 0, alpha=0.1, color=BLUE)
ax.fill_between(grouped['cycles_post_idp'], grouped['lstm_mean_error'], 0, alpha=0.1, color=AMBER)
ax.set_title(f'Q2: Prediction Error in the {POST_IDP_WINDOW}-Cycle Post-IDP Window\n'
             'Positive = late prediction (overestimates remaining life)', fontsize=12)
ax.set_xlabel('Cycles after IDP')
ax.set_ylabel('Mean prediction error (cycles)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

---
## Q3 — Fleet-level SHAP: which features drive RUL predictions?

In [ ]:
# ============================================================
# Q3: Fleet-level SHAP on XGBoost (TreeSHAP — exact)
# Source: Lundberg & Lee (2017). A Unified Approach to Interpreting
#         Model Predictions. NeurIPS.
# ============================================================

print('Computing SHAP values for XGBoost (fleet-level)...')
explainer  = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

print(f'SHAP values computed. Shape: {shap_values.shape}')
print()

mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance_df = pd.DataFrame({
    'feature':       ALL_FEATURES,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False)

print('Physics check:')
print('  Expected top features (GPA influence coefficients): HPT_degradation dominant')
print('  Actual top 5 features:')
for _, row in shap_importance_df.head(5).iterrows():
    print(f'    {row["feature"]:<30} mean |SHAP| = {row["mean_abs_shap"]:,.1f} cycles')
print()
print('If HPT_degradation leads by a large margin, the model learned the correct physics.')

In [ ]:
# Q3 Chart — feature importance bar chart (top 20)

top20 = shap_importance_df.head(20)

def feature_colour(fname):
    if fname.endswith('_norm'): return BLUE
    if fname in FEATURES_ISA:  return TEAL
    if fname in FEATURES_DEG:  return RED
    if fname in FEATURES_CTX:  return GREEN
    return PURPLE

colours = [feature_colour(f) for f in top20['feature']]

fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(top20['feature'][::-1], top20['mean_abs_shap'][::-1],
        color=colours[::-1], edgecolor='#0f1117')
ax.set_title('Q3: Fleet-Level Feature Importance (Mean Absolute SHAP Value)\n'
             'Blue=normalised sensors | Teal=ISA-corrected | Red=degradation | '
             'Green=contextual | Purple=categorical', fontsize=11)
ax.set_xlabel('Mean |SHAP| value (cycles of RUL impact)')
ax.grid(True, alpha=0.2, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Q3 SHAP beeswarm — shows direction of each feature's impact

shap_explanation = shap.Explanation(
    values      = shap_values,
    base_values = explainer.expected_value,
    data        = X_test.values,
    feature_names = ALL_FEATURES
)

plt.figure(figsize=(10, 8))
shap.plots.beeswarm(shap_explanation, max_display=15, show=True)
print('Each dot = one test row. Red = high feature value, Blue = low.')
print('Right of zero = pushed RUL prediction higher. Left = pushed it lower.')

---
## Q4 — Per-engine SHAP: diagnostic triage

In [ ]:
# ============================================================
# Q4: Per-engine SHAP — each engine's fault signature
# ============================================================

per_engine_shap = {}

for engine_id in TEST_ENGINES:
    engine_mask = (test_df['engine_id'] == engine_id).values
    engine_shap = shap_values[engine_mask]
    per_engine_shap[engine_id] = pd.Series(
        np.abs(engine_shap).mean(axis=0), index=ALL_FEATURES
    )

per_engine_df = pd.DataFrame(per_engine_shap).T

print('=== Q4: Per-Engine Top 3 SHAP Drivers (Diagnostic Triage) ===')
print()
for engine_id in TEST_ENGINES:
    top3 = per_engine_df.loc[engine_id].sort_values(ascending=False).head(3)
    last = test_df[test_df['engine_id'] == engine_id].iloc[-1]
    rag  = 'RED' if (last['EGT_margin'] < 30 or last['RUL'] < 200) else \
           'AMBER' if (last['EGT_margin'] < 50 or last['RUL'] < 500) else 'GREEN'
    print(f'{engine_id} [{rag}]:')
    for feat, val in top3.items():
        print(f'    {feat:<30} {val:,.1f} cycles impact')
    print()

In [ ]:
# Q4 Heatmap — per-engine SHAP across top 15 features

top15_features = shap_importance_df['feature'].head(15).tolist()
heatmap_data   = per_engine_df[top15_features]

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(heatmap_data, ax=ax, cmap='YlOrRd', annot=True, fmt='.0f',
            linewidths=0.5, cbar_kws={'label': 'Mean |SHAP| (cycles)'})
ax.set_title('Q4: Per-Engine SHAP Feature Importance — Top 15 Features\n'
             'Brighter = feature contributes more to that engine\'s RUL prediction',
             fontsize=11)
ax.set_xlabel('Feature (ranked left to right by fleet importance)')
ax.set_ylabel('Engine')
plt.tight_layout()
plt.show()
print('A bright cell in an unusual column = that system needs attention on that engine.')
print('This is the diagnostic triage value of per-engine SHAP.')

---
## Q5 — Do engines that start with lower EGT margin degrade faster?

In [ ]:
# ============================================================
# Q5: Initial EGT margin vs degradation rate — full fleet
# ============================================================

# --------------------------------------------------------
# EQUATION REF — Degradation rate via linear regression
# numpy: np.polyfit(x, y, deg=1) returns [slope, intercept]
#
# slope = rate of EGT margin change per cycle
# We negate and multiply by 1000 to get: degrees lost per 1,000 EFC
#
# To verify in Excel: =SLOPE(EGT_margin_values, cycle_values) for one engine
#
# EQUATION REF — Pearson correlation
# numpy: np.corrcoef(x, y)[0, 1]
# r in [-1, 1]. r ~ 0 = no correlation.
# --------------------------------------------------------

q5_stats = []

for engine_id in all_engines:
    eng            = df[df['engine_id'] == engine_id].copy()
    initial_margin = eng['EGT_margin'].values[:200].mean()
    slope, _       = np.polyfit(eng['cycle'].values, eng['EGT_margin'].values, deg=1)
    deg_rate       = -slope * 1000   # positive = margin falling = degrading

    last = eng.iloc[-1]
    rag  = 'RED' if (last['EGT_margin'] < 30 or last['RUL'] < 200) else \
           'AMBER' if (last['EGT_margin'] < 50 or last['RUL'] < 500) else 'GREEN'

    q5_stats.append({
        'engine_id':         engine_id,
        'initial_margin':    round(initial_margin, 2),
        'deg_rate_per_1000': round(deg_rate, 3),
        'rag':               rag
    })

q5_df       = pd.DataFrame(q5_stats)
correlation = np.corrcoef(q5_df['initial_margin'], q5_df['deg_rate_per_1000'])[0, 1]

print('=== Q5: Initial EGT Margin vs Degradation Rate ===')
print()
print(f'{"Engine":<12} {"Initial Margin":>16} {"Deg Rate /1000 EFC":>20} {"RAG":>6}')
print('-' * 58)
for _, row in q5_df.sort_values('initial_margin').iterrows():
    print(f'{row["engine_id"]:<12} {row["initial_margin"]:>16.2f}°C '
          f'{row["deg_rate_per_1000"]:>18.3f}°C  {row["rag"]:>6}')
print()
print(f'Pearson correlation (initial margin vs degradation rate): r = {correlation:.4f}')
if correlation < -0.3:
    print('  Negative correlation: engines with lower initial margin tend to degrade faster.')
elif abs(correlation) < 0.3:
    print('  Near-zero correlation: initial margin and degradation rate are largely independent.')
else:
    print('  Positive correlation: engines with higher initial margin degrade faster — investigate.')

In [ ]:
# Q5 Chart — scatter plot with trend line

rag_colours = {'RED': RED, 'AMBER': AMBER, 'GREEN': GREEN}

fig, ax = plt.subplots(figsize=(9, 6))
for _, row in q5_df.iterrows():
    ax.scatter(row['initial_margin'], row['deg_rate_per_1000'],
               color=rag_colours[row['rag']], s=80, zorder=5)
    ax.annotate(row['engine_id'], (row['initial_margin'], row['deg_rate_per_1000']),
                textcoords='offset points', xytext=(6, 4), fontsize=7, alpha=0.85)

z      = np.polyfit(q5_df['initial_margin'], q5_df['deg_rate_per_1000'], 1)
p      = np.poly1d(z)
x_line = np.linspace(q5_df['initial_margin'].min() - 1, q5_df['initial_margin'].max() + 1, 50)
ax.plot(x_line, p(x_line), color='white', linestyle='--', alpha=0.5,
        linewidth=1.5, label=f'Trend line (r = {correlation:.3f})')

for rag, col in rag_colours.items():
    ax.scatter([], [], color=col, label=rag, s=80)

ax.set_title('Q5: Does Starting EGT Margin Predict Degradation Rate?\nEach point = one engine',
             fontsize=12)
ax.set_xlabel('Initial EGT Margin (°C) — mean of first 200 cycles')
ax.set_ylabel('Degradation Rate (°C margin lost per 1,000 EFC)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()
print(f'r = {correlation:.4f} — near-zero: starting condition and degradation rate are independent.')
print('This is a property of the Phase 1 data generator (they were sampled independently).')
print('In real fleets, hot-and-high routes may produce both lower initial margin AND faster degradation.')

---
## Audit 4 — Verification Summary

All four checks are verified by the printed outputs in earlier cells. This cell summarises what was found.

In [ ]:
# ============================================================
# AUDIT 4 — Summary of all four verification checks
# ============================================================

print('=' * 65)
print('AUDIT 4 — Phase 4 Verification Summary')
print('=' * 65)
print()

# A4-1: Baseline normalisation
e1_mean_a, e1_std_a = baseline_stats['ENG-001']['EGT']
e1_raw_a   = df[df['engine_id'] == 'ENG-001']['EGT'].values[0]
e1_norm_a  = df[df['engine_id'] == 'ENG-001']['EGT_norm'].values[0]
e1_manual_a = (e1_raw_a - e1_mean_a) / e1_std_a
a41_pass = abs(e1_manual_a - e1_norm_a) < 0.0001

print('A4-1: Baseline normalisation (ENG-001, EGT feature)')
print(f'  Formula: (raw - baseline_mean) / baseline_std')
print(f'  Baseline mean: {e1_mean_a:.4f}  |  Baseline std: {e1_std_a:.4f}')
print(f'  Raw value at row 0: {e1_raw_a:.4f}')
print(f'  Manual result:  {e1_manual_a:.4f}')
print(f'  Column result:  {e1_norm_a:.4f}')
print(f'  RESULT: {"PASS" if a41_pass else "FAIL"}')
print(f'  To verify in Excel: filter ENG-001, AVERAGE/STDEV of first 200 EGT rows,')
print(f'  then =({e1_raw_a:.4f} - {e1_mean_a:.4f}) / {e1_std_a:.4f}')
print()

# A4-2: Piecewise RUL target
eng4_check   = df[df['engine_id'] == 'ENG-004'].copy()
eng4_alarm_a = cusum_alarm_cycles['ENG-004']
eng4_cap_a   = eng4_check[eng4_check['cycle'] == eng4_alarm_a]['RUL'].values[0]
eng4_pre_u   = eng4_check[eng4_check['cycle'] < eng4_alarm_a]['RUL_target'].nunique()
a42_pass     = (eng4_pre_u == 1)

print('A4-2: Piecewise RUL target (ENG-004)')
print(f'  Rule: pre-IDP rows all share the same RUL_target (the cap value)')
print(f'  CUSUM alarm cycle: {int(eng4_alarm_a):,}')
print(f'  RUL cap value:     {int(eng4_cap_a):,}')
print(f'  Unique RUL_target values pre-IDP: {eng4_pre_u}  (should be 1)')
print(f'  RESULT: {"PASS" if a42_pass else "FAIL"}')
print()

# A4-3: post_IDP flag alignment
eng4_first_post_a = df[(df['engine_id'] == 'ENG-004') & (df['post_IDP'] == 1)]['cycle'].min()
a43_pass = (eng4_first_post_a == eng4_alarm_a)

print('A4-3: post_IDP flag alignment (ENG-004)')
print(f'  Rule: first row with post_IDP=1 should equal CUSUM alarm cycle')
print(f'  First post_IDP=1 cycle: {int(eng4_first_post_a):,}')
print(f'  CUSUM alarm cycle:      {int(eng4_alarm_a):,}')
print(f'  RESULT: {"PASS" if a43_pass else "FAIL"}')
print()

# A4-4: PHM score formula
late_penalty  = np.exp(np.clip(100 / 10, -500, 500)) - 1
early_penalty = np.exp(np.clip(100 / 13, -500, 500)) - 1
exact_penalty = np.exp(np.clip(0   / 10, -500, 500)) - 1
expected_total = late_penalty + early_penalty + exact_penalty
computed_total = compute_phm_score([1000, 1000, 1000], [1100, 900, 1000])
a44_pass = abs(expected_total - computed_total) < 0.01

print('A4-4: PHM score formula (three test cases)')
print(f'  Source: Saxena et al. (2008), Section 5')
print(f'  error=+100 (late):  exp(100/10) - 1 = {late_penalty:,.2f}')
print(f'  error=-100 (early): exp(100/13) - 1 = {early_penalty:,.2f}')
print(f'  error=   0 (exact): exp(0)      - 1 = {exact_penalty:.2f}')
print(f'  Expected total: {expected_total:,.2f}')
print(f'  Computed total: {computed_total:,.2f}')
print(f'  RESULT: {"PASS" if a44_pass else "FAIL"}')
print()

all_pass = all([a41_pass, a42_pass, a43_pass, a44_pass])
print('=' * 65)
print(f'AUDIT 4 OVERALL: {"ALL CHECKS PASSED" if all_pass else "ONE OR MORE CHECKS FAILED — REVIEW ABOVE"}')
print('=' * 65)

---
## Phase 4 Summary

| Question | Key finding |
|---|---|
| Q1 — Accuracy | XGBoost outperformed LSTM on this dataset. Late-life errors are the operationally critical window. |
| Q2 — Post-IDP window | XGBoost more accurate on all four test engines in the 200 cycles after IDP. LSTM's temporal memory did not pay off at this dataset size. |
| Q3 — Fleet SHAP | HPT_degradation dominates predictions — six times higher than the next feature. Matches GPA physics exactly. |
| Q4 — Per-engine SHAP | HPT_degradation is the top driver for all test engines. Secondary drivers vary — providing a per-engine fault signature for maintenance triage. |
| Q5 — Initial condition | Near-zero correlation (r ≈ 0.02). Starting condition and degradation rate are independent in this dataset. |

---

## Known limitations

1. **Clean data overstates accuracy** — no OAT scatter in EGT margin (Assumption A-02). Real data has ±3–5°C residual scatter.
2. **Run-to-threshold, not run-to-failure** — RUL defined as cycles to EGT redline, not actual failure. C-MAPSS uses run-to-failure.
3. **LSTM needs more data** — with ~1,750 cycles per engine and 16 training engines, LSTM cannot fully exploit temporal patterns. C-MAPSS has hundreds of engines.
4. **SHAP on XGBoost only** — TreeSHAP is exact. LSTM SHAP (DeepSHAP) is approximate and was not implemented.
5. **All engines share the same degradation mode** — real fleets show more diverse failure signatures.

---
*Phase 4 complete. Audit 4 passed. Next: Phase 5 — GitHub, LinkedIn, David outreach.*